# Lab 6: API Management para IA (10 min)

## Objetivos
- Entender o papel do APIM como gateway para modelos de IA
- Ver como APIM protege e gere os endpoints
- Testar um endpoint via APIM (se configurado)
- Compreender políticas (rate limiting, caching, token limits)

## 📖 Porquê APIM para IA?

O **Azure API Management** funciona como gateway entre as tuas aplicações e os modelos de IA:

```
App/Frontend         APIM                     Azure OpenAI
┌──────────┐    ┌──────────────┐    ┌─────────────────────┐
│           │───►│ Rate Limit    │───►│ GPT-4o (East US)    │
│  A tua    │    │ Auth/Keys     │    ├─────────────────────┤
│  App      │◄───│ Caching       │◄───│ GPT-4o (West EU)    │
│           │    │ Load Balance  │    │ (failover)          │
│           │    │ Logging       │    └─────────────────────┘
└──────────┘    │ Token Limits  │
                │ Content Safety│
                └──────────────┘
```

### Benefícios

| Funcionalidade | Descrição |
|---------------|-----------|
| **Rate Limiting** | Limitar nº de requests/tokens por utilizador |
| **Load Balancing** | Distribuir tráfego entre múltiplos deployments |
| **Caching** | Cache semântico para respostas repetidas |
| **Auth & Keys** | Gerir chaves de acesso centralmente |
| **Monitoring** | Logs centralizados de todos os pedidos |
| **Content Safety** | Filtros de conteúdo antes de chegar ao modelo |
| **Token Metrics** | Rastrear consumo de tokens por app/utilizador |

## 🖥️ Exercício 6.1: Explorar APIM no Portal (Demonstração)

Se o instrutor tiver um APIM configurado:

1. Vai ao [Portal Azure](https://portal.azure.com) → API Management
2. Observa:
   - **APIs** registadas (Azure OpenAI API)
   - **Products** (agrupamento de APIs)
   - **Subscriptions** (chaves para consumidores)
   - **Policies** (XML de políticas)

### Exemplo de Política de Rate Limiting por Tokens

```xml
<policies>
    <inbound>
        <!-- Limitar a 10000 tokens por minuto por subscription -->
        <azure-openai-token-limit
            counter-key="@(context.Subscription.Id)"
            tokens-per-minute="10000"
            estimate-prompt-tokens="true" />
        
        <!-- Cache semântico -->
        <azure-openai-semantic-cache-lookup
            score-threshold="0.9"
            embeddings-backend-id="embeddings-backend" />
    </inbound>
</policies>
```

In [ ]:
# Setup
from dotenv import load_dotenv
import os
import requests

load_dotenv("../.env")

# Verificar se APIM está configurado
apim_endpoint = os.getenv("APIM_ENDPOINT")
apim_key = os.getenv("APIM_SUBSCRIPTION_KEY")

if apim_endpoint and not apim_endpoint.startswith("<"):
    print(f"✅ APIM configurado: {apim_endpoint}")
else:
    print("⚠️ APIM não configurado - vamos simular o conceito com Azure OpenAI direto")
    print("   Em produção, terias o APIM à frente como gateway")

## 💻 Exercício 6.2: Chamar modelo via APIM

Se tiveres APIM configurado, a chamada é igual — só muda o endpoint e a chave!

In [ ]:
from openai import AzureOpenAI

if apim_endpoint and not apim_endpoint.startswith("<"):
    # Chamar via APIM
    apim_client = AzureOpenAI(
        azure_endpoint=apim_endpoint,
        api_key=apim_key,
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
    )
    print("🔄 A chamar via APIM...")
else:
    # Fallback para Azure OpenAI direto
    apim_client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
    )
    print("🔄 A chamar Azure OpenAI direto (sem APIM)...")

response = apim_client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    messages=[{"role": "user", "content": "Diz uma curiosidade sobre Portugal em 1 frase."}],
    max_tokens=100,
)

print(f"\n🤖 {response.choices[0].message.content}")
print(f"📊 Tokens: {response.usage.total_tokens}")

## 💻 Exercício 6.3: Simular Rate Limiting

Vamos simular o que acontece quando um utilizador faz muitos pedidos rapidamente.

In [ ]:
import time

# Simular múltiplos pedidos rápidos
print("🔄 A enviar 5 pedidos rápidos seguidos...\n")

for i in range(5):
    try:
        inicio = time.time()
        r = apim_client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
            messages=[{"role": "user", "content": f"Diz o número {i+1} em português."}],
            max_tokens=10,
        )
        duracao = time.time() - inicio
        print(f"  ✅ Pedido {i+1}: {r.choices[0].message.content.strip()} ({duracao:.2f}s, {r.usage.total_tokens} tokens)")
    except Exception as e:
        print(f"  ❌ Pedido {i+1}: Rate limited! {e}")

print("\n💡 Com APIM, poderias limitar a 3 pedidos/minuto por utilizador!")

## 📖 Padrão Recomendado em Produção

```
┌─────────────────────────────────────────────────────┐
│                    APIM Gateway                      │
│                                                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────────────┐  │
│  │ Auth &   │  │ Rate     │  │ Load Balancer     │  │
│  │ API Keys │→ │ Limiting │→ │                   │  │
│  └──────────┘  └──────────┘  │ ┌──────────────┐ │  │
│                               │ │ OpenAI East  │ │  │
│  ┌──────────┐  ┌──────────┐  │ ├──────────────┤ │  │
│  │ Semantic │  │ Token    │  │ │ OpenAI West  │ │  │
│  │ Cache    │  │ Metrics  │  │ ├──────────────┤ │  │
│  └──────────┘  └──────────┘  │ │ OpenAI North │ │  │
│                               │ └──────────────┘ │  │
│                               └──────────────────┘  │
└─────────────────────────────────────────────────────┘
```

### Vantagem: Uma chave para o front-end, múltiplos backends
- O front-end nunca vê as chaves do Azure OpenAI
- Se um backend falhar, o APIM faz failover automático
- Métricas centralizadas de consumo

## ✅ Resumo

Neste lab aprendeste:
- O papel do APIM como gateway para IA
- Políticas de rate limiting, caching e token limits
- Como chamar modelos via APIM (mesmo SDK!)
- Padrão recomendado para produção

**Próximo:** [Lab 7 - Observabilidade](lab07-observabilidade.ipynb)